<a href="https://colab.research.google.com/github/Sambradshaw19011/CSE-450/blob/main/module5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models

# ----------------------------
# 1. Paths
# ----------------------------
# Update this path to wherever your unzipped training folders are
# Option A: if you combined training1 and training2 into one folder called "train"
data_dir = "train"

# Example structure:
# train/
#   0000/
#   0001/
#   ...
#   0042/

# ----------------------------
# 2. Settings
# ----------------------------
img_height = 48
img_width = 48
batch_size = 32
num_classes = 43
seed = 42

# ----------------------------
# 3. Load datasets
# ----------------------------
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=seed,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode="int"
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=seed,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode="int"
)

class_names = train_ds.class_names
print("Classes:", class_names)
print("Number of classes:", len(class_names))

# ----------------------------
# 4. Improve performance
# ----------------------------
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)

# ----------------------------
# 5. Optional data augmentation
# ----------------------------
data_augmentation = tf.keras.Sequential([
    layers.RandomZoom(0.10),
    layers.RandomRotation(0.05),
    layers.RandomTranslation(0.05, 0.05),
])

# ----------------------------
# 6. Build custom CNN
# ----------------------------
model = models.Sequential([
    layers.Input(shape=(img_height, img_width, 3)),

    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(32, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(256, (3, 3), padding="same", activation="relu"),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.4),
    layers.Dense(num_classes)
])

model.summary()

# ----------------------------
# 7. Compile
# ----------------------------
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)

# ----------------------------
# 8. Callbacks
# ----------------------------
early_stop = tf.keras.callbacks.EarlyStopping(
    monitor="val_accuracy",
    patience=5,
    restore_best_weights=True
)

# ----------------------------
# 9. Train
# ----------------------------
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
    callbacks=[early_stop]
)

# ----------------------------
# 10. Plot training history
# ----------------------------
plt.figure(figsize=(8,5))
plt.plot(history.history["accuracy"], label="train accuracy")
plt.plot(history.history["val_accuracy"], label="val accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.title("Training History")
plt.show()

# ----------------------------
# 11. Evaluate
# ----------------------------
val_loss, val_acc = model.evaluate(val_ds)
print("Validation Loss:", val_loss)
print("Validation Accuracy:", val_acc)